In [ ]:
import os
import sys
import jax

jax.config.update("jax_enable_x64", True)

import numpy as np
import qutip as qt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.config import ExperimentConfig
from src.control.waveforms import (
    build_bell_waveforms,
    build_local_drive_operators,
    pack_sequence,
)
from src.calibration.virtual_z_cal import calibrate_virtual_z
from src.control.virtual_z import apply_frame
from src.devices.qutip_device import HiddenDevice
from src.experiments.chsh import run_chsh_experiment
from src.analysis.plots import plot_chsh_results
from src.physics.operators import assemble_H0, build_two_qubit_base

# Load configuration and calibrated artifacts
config = ExperimentConfig.load("configs/two_qubit_CZ.yaml")
dt = config.pulse.dt_ns
duration_ns = 20.0

cal_pulse_rad = np.load("../artifacts/cal_pulse_rad.npy")
cal_g, cal_d, cal_a = np.load("../artifacts/cal_params.npy")

# Calibrate Virtual-Z phases on the JAX Digital Twin
H_anharm, H_int_base, Hz_control = build_two_qubit_base(
    levels=config.hardware.levels
)
Hx1, Hy1, Hx2, Hy2 = build_local_drive_operators(levels=config.hardware.levels)
H0_cal = assemble_H0(H_anharm, H_int_base, Hz_control, cal_g, cal_d, cal_a)

vz_cal = calibrate_virtual_z(
    duration_ns,
    dt,
    cal_a,
    cal_d,
    cal_pulse_rad,
    H0_cal,
    Hx1,
    Hy1,
    Hx2,
    Hy2,
    Hz_control,
)
c1, c2 = vz_cal["v_phase1"], vz_cal["v_phase2"]

# 3. Play the Bell state sequence on the noisy QuTiP Fridge
fridge = HiddenDevice(
    true_coupling=config.hardware.coupling_mhz,
    true_detuning=config.hardware.idle_detuning_mhz,
    true_anharm=config.hardware.anharmonicity_mhz,
    t1_ns=config.decoherence.t1_ns,
    t2_ns=config.decoherence.t2_ns,
)
rho00 = qt.ket2dm(qt.tensor(qt.basis(3, 0), qt.basis(3, 0)))
wf = build_bell_waveforms(
    duration_ns, dt, cal_a, cal_d, cal_pulse_rad, v_phase2=c2
)

rho_f = fridge.play_sequence(pack_sequence(wf), rho00)
rho_framed = apply_frame(rho_f, c1, c2)
print(
    f"Fridge Bell state prepared! (Virtual-Z phases: Q1={c1:.2f} rad,"
    f" Q2={c2:.2f} rad)"
)

In [ ]:
# Sanity Check on Ideal |Phi+> State
phi_ideal = (
    qt.tensor(qt.basis(3, 0), qt.basis(3, 0))
    + qt.tensor(qt.basis(3, 1), qt.basis(3, 1))
).unit()
S_ideal, err_ideal, _ = run_chsh_experiment(
    qt.ket2dm(phi_ideal), n_shots=5000, seed=42
)
print(
    f"1. Ideal |Phi+> Sanity Check     -> S = {S_ideal:.3f} ±"
    f" {err_ideal:.3f} (Quantum max: {2*np.sqrt(2):.3f})"
)

# Fridge State WITH Virtual-Z Frame Correction
S_cal, err_cal, E_cal = run_chsh_experiment(rho_framed, n_shots=5000, seed=42)
sigma_violation = (S_cal - 2.0) / err_cal
print(
    f"2. Fridge (Calibrated Virtual-Z) -> S = {S_cal:.3f} ±"
    f" {err_cal:.3f} ({sigma_violation:.1f}σ violation over classical"
    " bound 2.0)"
)

# Fridge State WITHOUT Virtual-Z Frame Correction
S_uncal, err_uncal, _ = run_chsh_experiment(rho_f, n_shots=5000, seed=42)
print(
    f"3. Fridge (Uncorrected Frame)    -> S = {S_uncal:.3f} ± {err_uncal:.3f}\n"
)

for k, v in E_cal.items():
    ideal_str = "-0.7071" if k == "E(a', b')" else "+0.7071"
    print(f"   {k:10s} = {v:+.4f}  (Ideal: {ideal_str})")

In [ ]:
fig = plot_chsh_results(
    E_cal,
    [S_uncal, S_cal, S_ideal],
    [err_uncal, err_cal, err_ideal],
)
fig.show()